# 06 — Channel Detection & Visualisation

**Author:** Zeineb Turki  
**Date:** April 2026  
**Phase:** 3.5 — Structure Validation  

## Objective

Detect and visualise every channel pattern in SPY daily data.
Channels use pivot + Theil-Sen regression (50-bar window), with R² ≥ 0.70,
containment ≥ 70%, and parallel slopes.

In [ ]:
import sys, os
from pathlib import Path

for mod_name in list(sys.modules.keys()):
    if mod_name.startswith('src'):
        del sys.modules[mod_name]

if 'google.colab' in str(getattr(sys, 'modules', {})) or os.path.exists('/content'):
    REPO_DIR  = '/content/regime-aware-ml-trading'
    PROJ_ROOT = os.path.join(REPO_DIR, 'regime-aware-ml-trading')
    if not os.path.isdir(PROJ_ROOT):
        os.system('git clone https://github.com/zaetae/regime-aware-ml-trading.git ' + REPO_DIR)
    else:
        os.system(f'cd {REPO_DIR} && git pull -q')
    os.system(f'{sys.executable} -m pip install -q yfinance hmmlearn scikit-learn seaborn statsmodels')
    _spy_path = os.path.join(PROJ_ROOT, 'data', 'raw', 'spy.csv')
    if not os.path.isfile(_spy_path):
        os.makedirs(os.path.dirname(_spy_path), exist_ok=True)
        import yfinance as yf
        import pandas as pd
        _spy = yf.download('SPY', start='2010-01-01', end='2026-01-01', auto_adjust=False)
        if isinstance(_spy.columns, pd.MultiIndex):
            _spy.columns = _spy.columns.droplevel(1)
        _spy = _spy[['Open', 'High', 'Low', 'Close', 'Volume']]
        if _spy.index.tz is not None:
            _spy.index = _spy.index.tz_localize(None)
        _spy.index.name = 'Date'
        _spy.to_csv(_spy_path)
        print(f"Downloaded SPY data to {_spy_path}")
else:
    def _find_project_root():
        current = Path.cwd()
        for _ in range(10):
            if (current / "src").is_dir():
                return current
            current = current.parent
        return Path.cwd().parent if (Path.cwd().parent / "src").is_dir() else Path.cwd()
    PROJ_ROOT = str(_find_project_root())

sys.path.insert(0, PROJ_ROOT)
os.chdir(PROJ_ROOT)

%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

from src.data.load_data import load_spy
from src.patterns.channels import detect_channel
from src.utils.plotting import plot_candlestick, add_trendline, add_event_marker

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

df = load_spy()
df_ch, ch_details = detect_channel(df, return_details=True)

print(f'Loaded {len(df)} bars: {df.index[0].date()} to {df.index[-1].date()}')
print(f'Channel detections: {len(ch_details)}')

In [ ]:
for i, d in enumerate(ch_details):
    print(f"{i+1}.  {d['event_date'].strftime('%Y-%m-%d')}  {d['pattern_type']}  "
          f"cr={d.get('containment_ratio',0):.0%}")

In [ ]:
FORWARD = 10

def plot_channel(det):
    """Plot one channel detection with trendlines."""
    start = det['start_idx']
    end = min(det['end_idx'] + FORWARD, len(df) - 1)
    chart = df.iloc[start:end + 1]
    window_slice = df.iloc[start:det['end_idx']]

    title = f"{det['pattern_type']}  —  {det['event_date'].strftime('%Y-%m-%d')}"
    if 'containment_ratio' in det:
        title += f"   cr={det['containment_ratio']:.0%}"

    fig, ax = plt.subplots(figsize=(14, 4.5))
    plot_candlestick(chart, ax=ax, title=title)

    add_trendline(ax, window_slice,
                  [det['upper_slope'], det['upper_intercept']],
                  det['window'], color='red', label='Upper trend')
    add_trendline(ax, window_slice,
                  [det['lower_slope'], det['lower_intercept']],
                  det['window'], color='blue', label='Lower trend')

    ep = df.loc[det['event_date'], 'Close']
    add_event_marker(ax, det['event_date'], ep,
                     marker='D', color='orange', size=80, label='Detection')
    ax.legend(fontsize=7, loc='upper left')
    fig.tight_layout()
    plt.show()

## Channel Charts

Each cell below plots one detected channel.

In [ ]:
if len(ch_details) > 0:
    plot_channel(ch_details[0])
else:
    print("No channels detected with current quality thresholds.")

## Conclusion

- Channels require strict quality gates: parallel slopes, R² ≥ 0.70, containment ≥ 70%
- Only the cleanest channel structures pass — low count is expected on daily SPY
- Each detection shows parallel trendlines with price respecting both boundaries